In [1]:
import torch
import numpy as np

# ============================================
# PyTorch mit MPS (Apple Silicon GPU) initialisieren
# ============================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ MPS (Apple Silicon GPU) ist verfügbar und wird verwendet!")
else:
    device = torch.device("cpu")
    print("⚠ MPS nicht verfügbar, nutze CPU")

print(f"Device: {device}")
print(f"PyTorch Version: {torch.__version__}\n")

✓ MPS (Apple Silicon GPU) ist verfügbar und wird verwendet!
Device: mps
PyTorch Version: 2.9.0



In [3]:
try:
    from datasets import load_dataset
    
    # Option 1: Wikipedia Deutsch (gute Qualität, mittelgroß)
    dataset_hf = load_dataset("wikimedia/wikipedia", "20231101.de", split="train", streaming=False)
    
    # Nimm die ersten N Artikel für schnelleres Training
    NUM_ARTICLES = 1000  # Kannst du erhöhen!
    dataset_hf = dataset_hf.select(range(min(NUM_ARTICLES, len(dataset_hf))))
    
    # Alle Texte kombinieren
    print(f"Verarbeite {len(dataset_hf)} Wikipedia-Artikel...")
    texts = [article['text'] for article in dataset_hf]
    text = ' '.join(texts)
    
    print(f"✓ Dataset geladen!")
    print(f"  Artikel: {NUM_ARTICLES}")
    print(f"  Zeichen: {len(text):,}")
    
except ImportError:
    print("⚠ 'datasets' nicht installiert. Installiere mit: pip install datasets")
    print("Nutze Fallback-Text...\n")
    text = "Transformer sind Deep-Learning-Modelle. " * 1000
except Exception as e:
    print(f"⚠ Fehler beim Laden: {e}")
    print("Nutze Fallback-Text...\n")
    text = "Transformer sind Deep-Learning-Modelle. " * 1000

Generating train split: 100%|██████████| 2845308/2845308 [00:06<00:00, 457647.71 examples/s]


Verarbeite 1000 Wikipedia-Artikel...
✓ Dataset geladen!
  Artikel: 1000
  Zeichen: 23,541,885


In [4]:
import tiktoken

encoding = tiktoken.get_encoding("gpt2")

# Text tokenisieren
tokens = encoding.encode(text)

print(f"✓ Text tokenisiert!")
print(f"  Anzahl Tokens: {len(tokens)}")
print(f"  Vocab Size: {encoding.n_vocab}")
print(f"\nErste 20 Token IDs: {tokens[:20]}")

# Zurück zu Text (zum Testen)
decoded_sample = encoding.decode(tokens[:20])
print(f"\nDekodierte erste 20 Tokens:\n'{decoded_sample}'")

# Tokens als PyTorch Tensor
token_tensor = torch.tensor(tokens, device=device)
print(f"\n✓ Tokens als Tensor erstellt!")
print(f"  Shape: {token_tensor.shape}")
print(f"  Device: {token_tensor.device}")

✓ Text tokenisiert!
  Anzahl Tokens: 8843709
  Vocab Size: 50257

Erste 20 Token IDs: [36235, 2439, 270, 21067, 2876, 4352, 435, 82, 49693, 463, 5177, 277, 25151, 304, 42326, 277, 1134, 83, 1469, 3310]

Dekodierte erste 20 Tokens:
'Alan Smithee steht als Pseudonym für einen fiktiven Reg'

✓ Tokens als Tensor erstellt!
  Shape: torch.Size([8843709])
  Device: mps:0


In [5]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """
    Dataset für Language Modeling:
    Input: Tokens [0, 1, 2, 3, 4]
    Target: Tokens [1, 2, 3, 4, 5]
    """
    def __init__(self, tokens, seq_length=128):
        self.tokens = tokens
        self.seq_length = seq_length
        self.num_sequences = (len(tokens) - 1) // seq_length
        
    def __len__(self):
        return self.num_sequences
    
    def __getitem__(self, idx):
        start_idx = idx * self.seq_length
        end_idx = start_idx + self.seq_length + 1
        
        sequence = self.tokens[start_idx:end_idx]
        
        # Input: alle außer letztes Token
        input_ids = torch.tensor(sequence[:-1], dtype=torch.long)
        # Target: alle außer erstes Token
        target_ids = torch.tensor(sequence[1:], dtype=torch.long)
        
        return input_ids, target_ids

In [44]:

SEQ_LENGTH = 128  # Länge jeder Sequenz
BATCH_SIZE = 48    # Anzahl Sequenzen pro Batch

dataset = TextDataset(tokens, seq_length=SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"✓ DataLoader neu erstellt: {len(dataloader)} Batches")

✓ DataLoader neu erstellt: 1439 Batches


In [30]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    Fügt Positions-Information zu den Embeddings hinzu.
    Transformer haben keine Ahnung von Wort-Reihenfolge ohne das!
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # Erstelle Positional Encoding Matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # Berechne die Sinusoide
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Gerade Indizes
        pe[:, 1::2] = torch.cos(position * div_term)  # Ungerade Indizes
        
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        x: [batch_size, seq_len, d_model]
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

In [31]:
class TokenEmbedding(nn.Module):
    """
    Kombiniert Token Embeddings mit Positional Encoding
    """
    def __init__(self, vocab_size, d_model, max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.d_model = d_model
        
    def forward(self, x):
        """
        x: [batch_size, seq_len] - Token IDs
        Returns: [batch_size, seq_len, d_model] - Embeddings
        """
        # Token IDs -> Vektoren
        x = self.embedding(x) * math.sqrt(self.d_model)
        
        # Positional Encoding hinzufügen
        x = self.pos_encoding(x)
        
        return x

In [32]:
VOCAB_SIZE = encoding.n_vocab  # ~50257 für GPT-2
D_MODEL = 512

# Embedding Layer erstellen
embedding_layer = TokenEmbedding(VOCAB_SIZE, D_MODEL).to(device)

print(f"✓ Embedding Layer erstellt!")
print(f"  Vocab Size: {VOCAB_SIZE}")
print(f"  Embedding Dimension: {D_MODEL}")
print(f"  Parameter: {sum(p.numel() for p in embedding_layer.parameters()):,}")



✓ Embedding Layer erstellt!
  Vocab Size: 50257
  Embedding Dimension: 512
  Parameter: 25,731,584


In [33]:
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Die Kern-Formel: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        Q: Query [batch, seq_len, d_k]
        K: Key [batch, seq_len, d_k]
        V: Value [batch, seq_len, d_k]
        mask: Optional [batch, seq_len, seq_len]
    
    Returns:
        output: [batch, seq_len, d_k]
        attention_weights: [batch, seq_len, seq_len]
    """
    d_k = Q.size(-1)
    
    # Schritt 1: QK^T - "Ähnlichkeit" zwischen Query und Key
    scores = torch.matmul(Q, K.transpose(-2, -1))  # [batch, seq_len, seq_len]
    
    # Schritt 2: Skalierung durch sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Schritt 3: Optional - Maske (z.B. für causale Attention)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Schritt 4: Softmax - Scores zu Wahrscheinlichkeiten
    attention_weights = F.softmax(scores, dim=-1)
    
    # Schritt 5: Gewichtete Summe der Values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

In [34]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention: Mehrere Attention-Heads arbeiten parallel
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        
        assert d_model % num_heads == 0, "d_model muss durch num_heads teilbar sein!"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension pro Head
        
        # Lineare Projektionen für Q, K, V (für ALLE Heads zusammen)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output Projektion (nach dem Konkatenieren)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def split_heads(self, x):
        """
        Teilt d_model in num_heads und d_k auf
        
        Input:  [batch, seq_len, d_model]
        Output: [batch, num_heads, seq_len, d_k]
        """
        batch_size, seq_len, d_model = x.size()
        
        # Reshape: [batch, seq_len, num_heads, d_k]
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        
        # Transpose: [batch, num_heads, seq_len, d_k]
        return x.transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Konkateniert alle Heads zurück
        
        Input:  [batch, num_heads, seq_len, d_k]
        Output: [batch, seq_len, d_model]
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        
        # Transpose: [batch, seq_len, num_heads, d_k]
        x = x.transpose(1, 2)
        
        # Reshape: [batch, seq_len, d_model]
        return x.contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, x, mask=None):
        """
        x: [batch, seq_len, d_model]
        Returns: [batch, seq_len, d_model], attention_weights
        """
        batch_size = x.size(0)
        
        # 1. Lineare Projektionen für Q, K, V
        Q = self.W_q(x)  # [batch, seq_len, d_model]
        K = self.W_k(x)  # [batch, seq_len, d_model]
        V = self.W_v(x)  # [batch, seq_len, d_model]
        
        # 2. Teile in mehrere Heads auf
        Q = self.split_heads(Q)  # [batch, num_heads, seq_len, d_k]
        K = self.split_heads(K)  # [batch, num_heads, seq_len, d_k]
        V = self.split_heads(V)  # [batch, num_heads, seq_len, d_k]
        
        # 3. Scaled Dot-Product Attention für alle Heads parallel
        attention_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        # attention_output: [batch, num_heads, seq_len, d_k]
        
        # 4. Konkateniere alle Heads
        attention_output = self.combine_heads(attention_output)
        # attention_output: [batch, seq_len, d_model]
        
        # 5. Finale lineare Projektion
        output = self.W_o(attention_output)
        
        # 6. Dropout
        output = self.dropout(output)
        
        return output, attention_weights


In [35]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network:
    FFN(x) = max(0, xW1 + b1)W2 + b2
    
    d_model -> d_ff -> d_model
    (512 -> 2048 -> 512)
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x

In [36]:
class DecoderBlock(nn.Module):
    """
    Transformer Decoder Block - mit MASKED Attention!
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Masked Multi-Head Attention
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Masked Attention + Residual + Norm
        attention_output, _ = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attention_output))
        
        # Feed-Forward + Residual + Norm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

In [37]:
class GPTDecoder(nn.Module):
    """
    GPT-Style Transformer Decoder (Decoder-only Architektur)
    """
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        
        self.embedding = TokenEmbedding(vocab_size, d_model)
        
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Language Model Head: d_model -> vocab_size
        self.lm_head = nn.Linear(d_model, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        x: [batch, seq_len] - Token IDs
        Returns: [batch, seq_len, vocab_size] - Logits für jedes Token
        """
        x = self.embedding(x)
        x = self.dropout(x)
        
        # Durch alle Decoder Blocks
        for decoder_block in self.decoder_blocks:
            x = decoder_block(x, mask)
        
        # Language Model Head
        logits = self.lm_head(x)
        
        return logits
    
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0):
        """
        Text generieren (autoregressive)
        """
        self.eval()
        
        for _ in range(max_new_tokens):
            # Causale Maske erstellen
            seq_len = input_ids.size(1)
            mask = create_causal_mask(seq_len, input_ids.device)
            
            # Forward pass
            with torch.no_grad():
                logits = self.forward(input_ids, mask)
            
            # Nimm das letzte Token
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            
            # Sample nächstes Token
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Füge zur Sequenz hinzu
            input_ids = torch.cat([input_ids, next_token], dim=1)
        
        return input_ids

In [38]:
NUM_HEADS = 8
D_FF = 2048
NUM_LAYERS = 6

model = GPTDecoder(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    dropout=0.1
).to(device)

print(f"✓ GPT Decoder erstellt!")
print(f"  d_model: {D_MODEL}")
print(f"  num_heads: {NUM_HEADS}")
print(f"  d_ff: {D_FF}")
print(f"  num_layers: {NUM_LAYERS}")
print(f"  vocab_size: {VOCAB_SIZE}")
print(f"  Parameter: {sum(p.numel() for p in model.parameters()):,}")

✓ GPT Decoder erstellt!
  d_model: 512
  num_heads: 8
  d_ff: 2048
  num_layers: 6
  vocab_size: 50257
  Parameter: 70,427,729


In [39]:
def create_causal_mask(seq_len, device):
    """
    Erstellt eine Maske, sodass Position i nur auf Positionen <= i schauen kann.
    Verhindert, dass der Decoder in die Zukunft schaut!
    
    Returns: [seq_len, seq_len] mit 1 für erlaubt, 0 für verboten
    """
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask  # Lower triangular matrix


In [40]:
def save_checkpoint(model, optimizer, epoch, loss, filepath):
    """
    Speichert Model + Optimizer + Training-Info
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'vocab_size': VOCAB_SIZE,
        'd_model': D_MODEL,
        'num_heads': NUM_HEADS,
        'd_ff': D_FF,
        'num_layers': NUM_LAYERS,
    }
    torch.save(checkpoint, filepath)
    print(f"  💾 Checkpoint gespeichert: {filepath}")


def load_checkpoint(filepath, model, optimizer=None):
    """
    Lädt Model + optional Optimizer
    """
    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    
    print(f"  📂 Checkpoint geladen: Epoche {epoch}, Loss {loss:.4f}")
    return epoch, loss

In [41]:
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam
learning_rate = 3e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epoch, loss = load_checkpoint('checkpoint_epoch_2.pt', model, optimizer)

print(f"✓ Loss: CrossEntropyLoss")
print(f"✓ Optimizer: Adam (lr={learning_rate})")

# ============================================
# 13. TRAINING LOOP
# ============================================

def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Trainiert eine Epoche
    """
    model.train()
    total_loss = 0
    
    for batch_idx, (input_ids, target_ids) in enumerate(dataloader):
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        # Causale Maske
        seq_len = input_ids.size(1)
        mask = create_causal_mask(seq_len, device)
        
        # Forward pass
        logits = model(input_ids, mask)
        
        # Loss berechnen
        # logits: [batch, seq_len, vocab_size]
        # target: [batch, seq_len]
        loss = criterion(logits.view(-1, VOCAB_SIZE), target_ids.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Progress
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    return avg_loss

  📂 Checkpoint geladen: Epoche 2, Loss 3.1202
✓ Loss: CrossEntropyLoss
✓ Optimizer: Adam (lr=0.0003)


In [45]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    print(f"Epoche {epoch + 1}/{NUM_EPOCHS}")
    
    avg_loss = train_epoch(model, dataloader, criterion, optimizer, device)
    
    print(f"→ Durchschnittlicher Loss: {avg_loss:.4f}\n")

        
    if (epoch + 1) % 2 == 0:  # Alle 2 Epochen
        save_checkpoint(model, optimizer, epoch + 1, avg_loss, f'checkpoint_epoch_{epoch + 1}.pt')
    
    # Test Generation nach jeder Epoche
    print("  Text Generation Test:")
    test_prompts = ["Die Geschichte", "Im Jahr", "Deutschland ist"]
    
    for start_text in test_prompts:
        start_tokens = encoding.encode(start_text)
        input_ids = torch.tensor([start_tokens], device=device)
        
        generated_ids = model.generate(input_ids, max_new_tokens=20, temperature=0.8)
        generated_text = encoding.decode(generated_ids[0].cpu().tolist())
        
        print(f"  '{generated_text}'")
    
    print(f"  '{generated_text}'\n")
    print("-" * 60)

Epoche 1/10
  Batch 10/1439, Loss: 2.8127
  Batch 20/1439, Loss: 2.7685
  Batch 30/1439, Loss: 2.7470
  Batch 40/1439, Loss: 2.8783
  Batch 50/1439, Loss: 2.7637
  Batch 60/1439, Loss: 2.7782
  Batch 70/1439, Loss: 2.6601
  Batch 80/1439, Loss: 2.6563
  Batch 90/1439, Loss: 2.7607
  Batch 100/1439, Loss: 2.8665
  Batch 110/1439, Loss: 2.8402
  Batch 120/1439, Loss: 2.7417
  Batch 130/1439, Loss: 2.8833
  Batch 140/1439, Loss: 2.8089
  Batch 150/1439, Loss: 2.8648
  Batch 160/1439, Loss: 2.8073
  Batch 170/1439, Loss: 2.8858
  Batch 180/1439, Loss: 2.8501
  Batch 190/1439, Loss: 2.7465
  Batch 200/1439, Loss: 2.7914
  Batch 210/1439, Loss: 2.8326
  Batch 220/1439, Loss: 2.8654
  Batch 230/1439, Loss: 2.7905
  Batch 240/1439, Loss: 2.7383
  Batch 250/1439, Loss: 2.7982
  Batch 260/1439, Loss: 2.7047
  Batch 270/1439, Loss: 2.8229
  Batch 280/1439, Loss: 2.6946
  Batch 290/1439, Loss: 2.7970
  Batch 300/1439, Loss: 2.7328
  Batch 310/1439, Loss: 2.8015
  Batch 320/1439, Loss: 2.7048
  Bat

KeyboardInterrupt: 

In [49]:
save_checkpoint(model, optimizer, epoch + 1, avg_loss, f'checkpoint_epoch_{epoch + 1}.pt')

  💾 Checkpoint gespeichert: checkpoint_epoch_2.pt


In [48]:
test_prompts = ["Die Geschichte", "Im Jahr", "Deutschland ist"]

for start_text in test_prompts:
    start_tokens = encoding.encode(start_text)
    input_ids = torch.tensor([start_tokens], device=device)
    
    generated_ids = model.generate(input_ids, max_new_tokens=50, temperature=0.8)
    generated_text = encoding.decode(generated_ids[0].cpu().tolist())
    
    print(f"  '{generated_text}'\n\n------------\n")

  'Die Geschichte der Sowjetunion vorgelegt wurden. Neben der Führung der Todesstrafe begann bei der Entsorgung eines Friedens.“

Aufgrund der Bürger'

------------

  'Im Jahr 1939 wurde das Attentat in eine Jugendorganisation einzigartig für die neueren Fünfte errichtet, die Bürger- und Einrichtungen'

------------

  'Deutschland ist eine Zweigstellung stärker beherrscht.

Die weltweit erste, unter anderem François Couper, das sowohl einige als auch die bekan'

------------

